# Semantic Bridge Analysis 

1. Pull the full UCSD Map of Science backbone using the existing `semantic_bridge.mapping.ucsd` helper.
2. Build the normal multi-layer backbone payload.
3. Filter the full UCSD backbone using the **documents and discovered topic keywords**, not a hard-coded groundwater seed list.
4. Use the filtered backbone to create `topic_mappings` and the network visualization.

That keeps the notebook general: changing the document corpus changes the filtered science backbone.


## 1. Load libraries


In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from collections import Counter
import re

import pandas as pd
import spacy
import nltk

from semantic_bridge.notebook import display as nbutils
from semantic_bridge import api as sbp

SPACY_MODEL_CANDIDATES = ["en_core_web_lg", "en_core_web_md", "en_core_web_sm"]

# The Day 2 code uses NLTK tokenizers in a few helper paths.
# punkt_tab is required by newer NLTK releases.
nltk.download("punkt_tab", quiet=True)


def get_safe_subprocess_cwd():
    """Return a valid directory for subprocess calls, even if Jupyter's CWD vanished."""
    try:
        cwd = Path.cwd()
        if cwd.exists():
            return cwd
    except FileNotFoundError:
        pass

    for fallback in (Path.home(), Path("/tmp")):
        if fallback.exists():
            return fallback

    raise OSError("Could not find a valid working directory for spaCy model installation.")


def load_or_install_spacy_model(model_names):
    """Load the first available spaCy model, installing one if needed."""
    for model_name in model_names:
        try:
            return spacy.load(model_name), model_name
        except OSError:
            pass

    install_errors = []
    install_cwd = get_safe_subprocess_cwd()

    for model_name in model_names:
        print(f"spaCy model {model_name} not found. Attempting install from {install_cwd}...")
        try:
            subprocess.run(
                [sys.executable, "-m", "spacy", "download", model_name],
                check=True,
                cwd=install_cwd,
            )
            return spacy.load(model_name), model_name
        except Exception as exc:
            install_errors.append(f"{model_name}: {exc}")

    raise OSError(
        "No spaCy English model could be loaded or installed. "
        + " | ".join(install_errors)
    )


nlp, loaded_model = load_or_install_spacy_model(SPACY_MODEL_CANDIDATES)
print(f"Libraries loaded successfully. spaCy model: {loaded_model}")


## 2. Configure the case study

Edit the values in the next cell directly — case study name, corpus folder, and any optional services (CKAN, LLM topic labeler, MINT). Leave a service's URL or model blank to skip it. The `OPENAI_API_KEY` is the only secret pulled from the environment; set it in your shell before launching Jupyter rather than typing it into the notebook.


In [ ]:
# =============================================================================
# Edit these values for your case study. Leave optional URLs as "" to skip.
# =============================================================================

CASE_STUDY_NAME = "Groundwater Management Area 12"
CORPUS_FOLDER   = "groundwater-management-area-12"   # data/<this folder>/{raw,cleaned}

# Optional: pull the corpus from a CKAN dataset (leave CKAN_URL = "" to use local files only)
CKAN_URL          = "https://ckan.tacc.utexas.edu"
CKAN_DATASET_NAME = "subsidence-groundwater-semantic-bridge-corpus"

# Optional: relabel topics with an OpenAI-compatible LLM (leave TOPIC_LABELER_MODEL = "" to skip)
TOPIC_LABELER_MODEL    = ""
TOPIC_LABELER_BASE_URL = "https://ai.tejas.tacc.utexas.edu"
TOPIC_LABELER_API_KEY  = os.getenv("OPENAI_API_KEY", "")   # set in the shell, not in the notebook

# Optional: MINT API (leave MINT_API_BASE_URL = "" to skip)
MINT_API_BASE_URL    = "https://api.models.mint.tacc.utexas.edu/v2.0.0"
MINT_USERNAME        = "mint@isi.edu"
MINT_SVO_PER_PAGE    = 200
MINT_SVO_MAX_PAGES   = 3
MINT_MODEL_PER_PAGE  = 100
MINT_MODEL_MAX_PAGES = 3
MINT_TOPIC_PER_PAGE  = 100
MINT_TOPIC_MAX_PAGES = 3

# =============================================================================
# Resolved paths (you rarely need to change anything below)
# =============================================================================
tutorial_dir   = nbutils.resolve_tutorial_dir()
corpus_dir     = sbp.ensure_data_directory(tutorial_dir / "data" / CORPUS_FOLDER)
raw_dir        = sbp.ensure_data_directory(corpus_dir / "raw")
cleaned_dir    = sbp.ensure_data_directory(corpus_dir / "cleaned")
analysis_data_dir = cleaned_dir if any(cleaned_dir.iterdir()) else raw_dir
OUTPUT_DIR     = sbp.ensure_data_directory(tutorial_dir / "outputs")

print(f"Tutorial directory: {tutorial_dir}")
print(f"Case study:         {CASE_STUDY_NAME}")
print(f"Local corpus:       {analysis_data_dir}")
print(f"Outputs:            {OUTPUT_DIR}")


## 3. Optional: load the corpus from CKAN

If CKAN is configured, the CKAN resources are cached locally and used as the active input directory. Otherwise the notebook uses the local `data/...` corpus.


In [29]:
if CKAN_URL and CKAN_DATASET_NAME:
    ckan_cache_dir = sbp.ensure_data_directory(corpus_dir / "ckan_cache")
    ckan_dataset = sbp.fetch_ckan_dataset(CKAN_URL, CKAN_DATASET_NAME)
    downloaded_resources = sbp.sync_ckan_resources_to_directory(
        ckan_dataset,
        ckan_cache_dir,
        base_url=CKAN_URL,
    )
    analysis_data_dir = ckan_cache_dir
    print(f"Loaded {len(downloaded_resources)} CKAN resources from {CKAN_DATASET_NAME}")
else:
    print("Using the local tutorial corpus directories.")

nbutils.print_runtime_paths(tutorial_dir, corpus_dir, raw_dir, cleaned_dir, analysis_data_dir)


Loaded 3 CKAN resources from subsidence-groundwater-semantic-bridge-corpus


**Runtime Paths**
- **Tutorial directory:** `/Users/wmobley/Documents/GitHub/DSO/DSO-Institute-2026/Day-2/Morning`
- **Corpus directory:** `/Users/wmobley/Documents/GitHub/DSO/DSO-Institute-2026/Day-2/Morning/data/groundwater-management-area-12`
- **Raw source directory:** `/Users/wmobley/Documents/GitHub/DSO/DSO-Institute-2026/Day-2/Morning/data/groundwater-management-area-12/raw`
- **Cleaned text directory:** `/Users/wmobley/Documents/GitHub/DSO/DSO-Institute-2026/Day-2/Morning/data/groundwater-management-area-12/cleaned`
- **Active input directory:** `/Users/wmobley/Documents/GitHub/DSO/DSO-Institute-2026/Day-2/Morning/data/groundwater-management-area-12/ckan_cache`

## 4. Load and inspect documents


In [30]:
transcripts = sbp.load_documents(analysis_data_dir)
nbutils.print_document_list(transcripts)


Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 27 0 (offset 0)
Ignoring wrong pointing object 29 0 (offset 0)


**Loaded Documents:** 9
- `GMA12_DFCExpRep_2021.pdf`
- `GMA12_DFCResolution_2021.pdf`
- `GMA12_DFC_2021.pdf`
- `GMA12_MAGsbyCounty_2021.pdf`
- `GMA12_MAGsbyGCD_2021.pdf`
- `GMA12_NonRelevant_CalvertBluff_2021.pdf`
- `harris-galveston-subsidence-district-regulatory-plan-amended-2021.pdf`
- `texas-tribune-groundwater-pumping-on-gulf-coast-leads-to-subsidence.txt`
- `texas-tribune-groundwater-rights-and-water-crisis.txt`

In [31]:
nbutils.print_document_previews(sbp.preview_documents(transcripts))


### `GMA12\_DFCExpRep\_2021.pdf`
```text
 
DESIRED FUTURE CONDITION EXPLANATORY REPORT FOR 
GROUNDWATER MANAGEMENT AREA 12 
 
 
 
This report was considered and approved by the member districts of Groundwater Management 
Area 12 on January 2...
```

### `GMA12\_DFCResolution\_2021.pdf`
```text

```

### `GMA12\_DFC\_2021.pdf`
```text
Groundwater Management Area (GMA) 12 
Desired Future Conditions 
2021 Joint Planning 
 
Page 1 of 3 
Adopted Desired Future Conditions for Relevant Aquifers (Sparta, Queen City, and Carrizo-Wilcox aqu...
```

### `GMA12\_MAGsbyCounty\_2021.pdf`
```text
Groundwater Management Area (GMA) 12 
Modeled Available Groundwater for Relevant Aquifers by County 
2021 Joint Planning 
 
Values from GAM Run 21-017 MAG: Modeled Available Groundwater for the Aquife...
```

### `GMA12\_MAGsbyGCD\_2021.pdf`
```text
Groundwater Management Area (GMA) 12 
Modeled Available Groundwater for Relevant Aquifers by Groundwater Conservation District (GCD) 
2021 Joint Planning 
 
Values from GAM Run 21-017 MAG: Modeled Ava...
```

### `GMA12\_NonRelevant\_CalvertBluff\_2021.pdf`
```text
Memorandum To:  Texas Water Development Board From:  GMA 12 Date:   May 5, 2022 Subject: Calvert Bluff Aquifer in Williamson County I. INTRODUCTION  The Texas Water Development Board, in its July 2013...
```

### `harris-galveston-subsidence-district-regulatory-plan-amended-2021.pdf`
```text
 
Regulatory Plan 
2013 
 
 
 
 
 
 
 
 
Harris-Galveston  
Subsidence District 
1660 West Bay Area Blvd. 
Friendswood, Texas 77546-2640 
(281) 486-1105 
www.subsidence.org 
 
 
 
Adopted:  January 9,...
```

### `texas-tribune-groundwater-pumping-on-gulf-coast-leads-to-subsidence.txt`
```text

Beneath the Surface
Groundwater Pumping Takes an Unexpected Toll on the Texas Gulf Coast
Unlike the rest of the state, the Texas Gulf Coast has been working for decades to reduce dependency on ground...
```

### `texas-tribune-groundwater-rights-and-water-crisis.txt`
```text

<h1>The one thing Texas won’t do to save its water supply</h1>
<p class="byline">by Jayme Lozano Carver and Yuriko Schumacher, The Texas Tribune <br />May 29, 2025</p>
<p><em><a href="https://www.tex...
```

In [32]:
stats_df = sbp.build_stats_table(transcripts)
print("Transcript statistics:\n")
print(stats_df.to_string(index=False))


Transcript statistics:

                                                                   File  Characters  Words  Sentences
                                               GMA12_DFCExpRep_2021.pdf      679141 101247       3022
                                           GMA12_DFCResolution_2021.pdf           0      0          0
                                                     GMA12_DFC_2021.pdf        4159    630          3
                                            GMA12_MAGsbyCounty_2021.pdf        7319   1161          5
                                               GMA12_MAGsbyGCD_2021.pdf       10780   1605          9
                                GMA12_NonRelevant_CalvertBluff_2021.pdf        6892   1062         42
  harris-galveston-subsidence-district-regulatory-plan-amended-2021.pdf       28767   4119        184
texas-tribune-groundwater-pumping-on-gulf-coast-leads-to-subsidence.txt       25152   3750        137
                  texas-tribune-groundwater-rights-and-wat

## 5. Configure stopwords and topic model parameters


In [33]:
CUSTOM_STOPWORDS = {
    # Common English stopwords
    "the", "and", "for", "are", "but", "not", "you", "all", "can",
    "her", "was", "one", "our", "out", "this", "that", "with", "have",
    "from", "they", "been", "were", "said", "what", "when", "your",
    "more", "will", "there", "their", "about", "which", "into", "than",
    "them", "would", "could", "should", "who", "has", "had", "how",
    "its", "may", "these", "some", "such", "only", "other", "any",
    "most", "also", "very", "even", "just", "like", "both", "each",
    "did", "does", "his", "she", "him", "well", "many", "much",
    "where", "here", "now", "then", "because", "before", "after",
    "through", "during", "without", "within", "being", "under", "over",
    "again", "further", "once", "why", "while", "same", "those", "own",
    "too", "off", "down", "upon", "between", "few", "above", "below",
    "doing", "an", "as", "at", "be", "by", "he", "if", "in", "is",
    "it", "me", "my", "no", "of", "on", "or", "so", "to", "up", "we",

    # Conversational fillers / analysis boilerplate
    "yeah", "okay", "um", "uh", "hmm", "oh", "ah",
    "know", "think", "going", "got", "get", "let",
    "see", "want", "make", "really", "lot", "kind",
    "sort", "thing", "things", "stuff", "actually",
    "basically", "literally", "probably", "maybe",
    "guess", "mean", "means", "supposed", "trying",
    "interviewer", "interviewee", "question", "answer",
    "ask", "asked", "asking", "tell", "told", "telling",
    "talk", "talked", "talking", "discuss", "discussed",
    "say", "says", "saying", "read", "reading",
}

n_topics = int(os.getenv("N_TOPICS", "5") or "5")
max_vocabulary = int(os.getenv("MAX_VOCABULARY", "200") or "200")
top_words_display = int(os.getenv("TOP_WORDS_DISPLAY", "12") or "12")

nbutils.print_topic_parameters(n_topics, max_vocabulary, top_words_display)


**Topic Model Parameters**
- **Topics:** 5
- **Max vocabulary:** 200
- **Display keywords per topic:** 12

## 6. Preprocess text and discover topics


In [34]:
processed_docs, doc_names = sbp.preprocess_documents(
    transcripts,
    custom_stopwords=CUSTOM_STOPWORDS,
)

print("Text preprocessing complete")
if processed_docs:
    print(f"Example: {processed_docs[0][:150]}...")

# Enable LLM topic labels only when credentials are present.
ENABLE_LLM_TOPIC_LABELS = bool(TOPIC_LABELER_MODEL and TOPIC_LABELER_API_KEY)
print(f"LLM topic labels enabled: {ENABLE_LLM_TOPIC_LABELS}")


Text preprocessing complete
Example: desired future condition explanatory report groundwater management area report considered approved member districts groundwater management area januar...
LLM topic labels enabled: False


In [35]:
topic_results = sbp.discover_topics(
    processed_docs,
    n_topics=n_topics,
    max_vocabulary=max_vocabulary,
    topic_keyword_count=top_words_display,
    custom_stopwords=CUSTOM_STOPWORDS,
)

doc_topic_dist = topic_results["doc_topic_dist"]
topics_info = topic_results["topics_info"]
feature_names = topic_results["feature_names"]

if ENABLE_LLM_TOPIC_LABELS:
    topics_info = sbp.relabel_topics_with_llm(
        topics_info=topics_info,
        doc_topic_dist=doc_topic_dist,
        doc_names=doc_names,
        documents=transcripts,
        model=TOPIC_LABELER_MODEL,
        api_key=TOPIC_LABELER_API_KEY,
        base_url=TOPIC_LABELER_BASE_URL or None,
    )

nbutils.print_topic_discovery_summary(topics_info, feature_names, top_words_display)


**Topic Discovery Summary**
- **Vocabulary size:** 200
- **Topics discovered:** 5

### Topic 1: surface water, water supply, year
- **Keywords:** surface water, water supply, year, supply, surface, counties, demand, plan, board, available, district, data

### Topic 2: carrizo, subsidence, water
- **Keywords:** carrizo, subsidence, water, aquifer, wilcox, carrizo wilcox, groundwater, county, brazos, bluff, calvert bluff, calvert

### Topic 3: water, state, groundwater
- **Keywords:** water, state, groundwater, texas, irrigation, districts, district, use, www, supply, used, aquifer

### Topic 4: regulatory, district, plan
- **Keywords:** regulatory, district, plan, groundwater, water, boundary, area, subsidence, demand, total, areas, county

### Topic 5: gcd, groundwater, carrizo
- **Keywords:** gcd, groundwater, carrizo, wilcox, aquifer, carrizo wilcox, gma, brazos, district, county, desired, desired future

In [36]:
# Optional topic distribution chart.
try:
    topic_df, topic_fig = sbp.plot_topic_distribution(doc_topic_dist, doc_names, topics_info, n_topics)
    topic_fig.show()
except Exception as exc:
    print(f"Skipping topic distribution plot: {type(exc).__name__}: {exc}")


## 7. Optional: recommend MINT queries for each topic

This section only runs if `MINT_API_BASE_URL` is configured.


In [37]:
if MINT_API_BASE_URL:
    print("Fetching MINT model candidates...")

    mint_model_candidates = sbp.fetch_mint_model_candidates(
        base_url=MINT_API_BASE_URL,
        username=MINT_USERNAME or "mint@isi.edu",
        per_page=MINT_TOPIC_PER_PAGE,
        max_pages=MINT_TOPIC_MAX_PAGES,
    )

    print(f"✓ Fetched {len(mint_model_candidates)} MINT model candidates")

    mint_topic_recommendations = sbp.recommend_mint_queries_for_topics(
        topics_info,
        mint_model_candidates,
    )

    mint_topic_df = pd.DataFrame(mint_topic_recommendations)
    mint_topic_df.to_csv(OUTPUT_DIR / "mint_topic_recommendations.csv", index=False)
    display(mint_topic_df)

else:
    mint_model_candidates = []
    mint_topic_recommendations = []
    print("MINT is not configured; skipping MINT topic recommendations.")

Fetching MINT model candidates...
✓ Fetched 352 MINT model candidates


,topic,label,description,query_domains,query_tags
0,Topic 1,"Topic 1: surface water, water supply, year",,"[Hydrology, Weather, Decision Support]","[management, Reclaimed water, water balance, H..."
1,Topic 2,"Topic 2: carrizo, subsidence, water",,[Hydrology],"[COMSOL; poroelastic; FEM; compaction, HydroGe..."
2,Topic 3,"Topic 3: water, state, groundwater",,"[Hydrology, Weather]","[Hydrology, Agriculture, Geospatial Database, ..."
3,Topic 4,"Topic 4: regulatory, district, plan",,"[Hydrology, Agriculture]","[groundwater transport, Groundwater;Contaminan..."
4,Topic 5,"Topic 5: gcd, groundwater, carrizo",,"[Hydrology, Weather]","[HydroGeoSphere; integrated; FEM; transport, M..."


## 8. Science backbone: pull full UCSD and filter it to this corpus

This is the patched section.

The notebook now uses this sequence:

1. Pull/cache the full UCSD Map of Science `.net` file.
2. Build the normal science-backbone payload, optionally merging ETO if `outputs/eto_map_export.csv` exists.
3. Collect filtering terms from `topics_info` and the loaded documents.
4. Score domains and subdisciplines against those terms.
5. Keep a smaller, case-relevant science backbone for mapping and visualization.

No groundwater-specific seed term list is required. The filter follows the current document corpus.


In [38]:
# Configuration for the patched backbone section.
USE_FULL_UCSD_BACKBONE = True
USE_ETO_EXPORT_IF_AVAILABLE = True
FILTER_UCSD_TO_CURRENT_CASE = True

KEEP_TOP_DOMAINS = 6
KEEP_TOP_SUBDISCIPLINES_PER_DOMAIN = 30

ETO_EXPORT_PATH = OUTPUT_DIR / "eto_map_export.csv"
UCSD_NET_LOCAL = OUTPUT_DIR / "UCSDmap_with_disciplines.net.txt"
SCIENCE_BACKBONE_PATH = OUTPUT_DIR / "science_backbone.json"


In [39]:
print("Building science backbone...\n")

if USE_FULL_UCSD_BACKBONE:
    base_layer, ucsd_stats = sbp.fetch_ucsd_science_backbone(
        cache_path=UCSD_NET_LOCAL,
        fallback_to_compact=True,
        return_stats=True,
    )
    print("Using UCSD full science backbone source")
    print(f"  Disciplines: {ucsd_stats.get('n_disciplines')}")
    print(f"  Assigned subdisciplines: {ucsd_stats.get('n_assigned_subdisciplines')}")
    if ucsd_stats.get("fallback"):
        print(f"  WARNING: UCSD fetch fell back to compact scaffold: {ucsd_stats.get('error')}")
else:
    base_layer = None
    ucsd_stats = {}
    print("Using compact built-in UCSD scaffold")

eto_cluster_df = None
if USE_ETO_EXPORT_IF_AVAILABLE and ETO_EXPORT_PATH.exists():
    eto_cluster_df = sbp.load_eto_cluster_export(ETO_EXPORT_PATH)
    print(f"Loaded ETO export: {ETO_EXPORT_PATH} ({len(eto_cluster_df)} rows)")
else:
    print(f"No ETO export found at {ETO_EXPORT_PATH}; continuing with UCSD layer only")

backbone_payload = sbp.build_science_backbone_payload(
    eto_cluster_df=eto_cluster_df,
    base_layer=base_layer,
    selected_layer="merged",
    prefer_ucsd_fields=True,
)

science_backbone_full = sbp.selected_science_backbone(backbone_payload)

if FILTER_UCSD_TO_CURRENT_CASE:
    science_backbone, backbone_filter_debug = sbp.filter_science_backbone_for_case(
        science_backbone_full,
        topics_info=topics_info,
        documents=transcripts,
        processed_docs=processed_docs,
        keep_top_domains=KEEP_TOP_DOMAINS,
        keep_top_subdisciplines=KEEP_TOP_SUBDISCIPLINES_PER_DOMAIN,
    )
    backbone_payload["layers"]["case_filtered"] = science_backbone
    backbone_payload["selected_layer"] = "case_filtered"
    print("\nFiltered UCSD backbone using document/topic terms")
else:
    science_backbone = science_backbone_full
    backbone_filter_debug = {}
    print("\nNo science-backbone filter applied")

sbp.write_science_backbone_payload(backbone_payload, SCIENCE_BACKBONE_PATH)

print(f"\nScience backbone written to: {SCIENCE_BACKBONE_PATH}")
print(f"Full domains: {len(science_backbone_full)}")
print(f"Active domains: {len(science_backbone)}")
print(f"Active layer: {backbone_payload['selected_layer']}")

for domain, node in science_backbone.items():
    print(
        f"  - {domain}: score={node.get('filter_score', 'n/a')}, "
        f"subdisciplines={len(node.get('subdisciplines', []))}, "
        f"matches={node.get('matched_filter_terms', [])[:8]}"
    )


Building science backbone...

Using UCSD full science backbone source
  Disciplines: 13
  Assigned subdisciplines: 554
No ETO export found at /Users/wmobley/Documents/GitHub/DSO/DSO-Institute-2026/Day-2/Morning/outputs/eto_map_export.csv; continuing with UCSD layer only

Filtered UCSD backbone using document/topic terms

Science backbone written to: /Users/wmobley/Documents/GitHub/DSO/DSO-Institute-2026/Day-2/Morning/outputs/science_backbone.json
Full domains: 13
Active domains: 6
Active layer: case_filtered
  - Social Sciences: score=9.6, subdisciplines=11, matches=['plan', 'gma', 'management', 'planning', 'development']
  - Chemical, Mechanical, & Civil Engineering: score=8.7, subdisciplines=9, matches=['water', 'management', 'development']
  - Biology: score=7.75, subdisciplines=8, matches=['plan', 'water', 'management', 'conservation']
  - Earth Sciences: score=7.55, subdisciplines=1, matches=['water', 'groundwater']
  - Electrical Engineering & Computer Science: score=6.6, subdisc

## 9. Map topics onto the filtered science backbone

`science_backbone` and `topic_mappings` are intentionally separate objects:

- `science_backbone` is the active domain/subdiscipline structure.
- `topic_mappings` is a list of topic-to-domain mapping records.


In [40]:
print("Mapping discovered topics onto the filtered science backbone...\n")

topic_mappings = sbp.make_topic_mappings_from_backbone(
    topics_info,
    science_backbone,
    domains_per_topic=3
)

topic_mappings_df = sbp.topic_mappings_table(topic_mappings)
topic_mappings_path = OUTPUT_DIR / "topic_mappings.csv"
topic_mappings_df.to_csv(topic_mappings_path, index=False)

print(f"Created {len(topic_mappings)} topic mappings")
print(f"Wrote: {topic_mappings_path}")
display(topic_mappings_df)


Mapping discovered topics onto the filtered science backbone...

Created 5 topic mappings
Wrote: /Users/wmobley/Documents/GitHub/DSO/DSO-Institute-2026/Day-2/Morning/outputs/topic_mappings.csv


,topic,topic_label,keywords,primary_domain,secondary_domain,candidate_domain_labels,mapping_score,matched_terms
0,Topic 1,"Topic 1: surface water, water supply, year","surface water, water supply, year, supply, sur...",Biology,Health Professionals,"Biology, Health Professionals, Social Sciences",2.55,"water, plan"
1,Topic 2,"Topic 2: carrizo, subsidence, water","carrizo, subsidence, water, aquifer, wilcox",Earth Sciences,Biology,"Earth Sciences, Biology, Chemical, Mechanical,...",3.35,"water, groundwater"
2,Topic 3,"Topic 3: water, state, groundwater","water, state, groundwater, texas, irrigation",Earth Sciences,Biology,"Earth Sciences, Biology, Chemical, Mechanical,...",3.70,"water, groundwater"
3,Topic 4,"Topic 4: regulatory, district, plan","regulatory, district, plan, groundwater, water",Biology,Earth Sciences,"Biology, Earth Sciences, Health Professionals",3.35,"plan, water"
4,Topic 5,"Topic 5: gcd, groundwater, carrizo","gcd, groundwater, carrizo, wilcox, aquifer",Earth Sciences,Social Sciences,"Earth Sciences, Social Sciences",1.85,groundwater


## 10. Visualize the filtered semantic bridge network


In [41]:
coverage_df = sbp.domain_coverage_table(science_backbone, topic_mappings)
display(coverage_df)

semantic_graph = sbp.build_semantic_bridge_graph(
    science_backbone,
    topic_mappings,
    max_subdisciplines_per_domain=25,
    include_candidate_domain_edges=True,
)

pyvis_graph_path = OUTPUT_DIR / "semantic_bridge_pyvis_graph_2d.html"

sbp.write_pyvis_semantic_graph(
    semantic_graph,
    pyvis_graph_path,
    height="780px",
    width="100%",
    notebook=True,
    cdn_resources="remote",
)

sbp.display_html_graph(pyvis_graph_path, width="100%", height=800)

,domain,filter_score,matched_filter_terms,primary_topic_count,candidate_topic_count,primary_topics,candidate_topics
3,Earth Sciences,7.55,"water, groundwater",3,4,"Topic 2: carrizo, subsidence, water; Topic 3: ...","Topic 2: carrizo, subsidence, water; Topic 3: ..."
2,Biology,7.75,"plan, water, management, conservation",2,4,"Topic 1: surface water, water supply, year; To...","Topic 1: surface water, water supply, year; To..."
0,Social Sciences,9.60,"plan, gma, management, planning, development",0,2,,"Topic 1: surface water, water supply, year; To..."
1,"Chemical, Mechanical, & Civil Engineering",8.70,"water, management, development",0,2,,"Topic 2: carrizo, subsidence, water; Topic 3: ..."
5,Health Professionals,4.40,"plan, management, joint",0,2,,"Topic 1: surface water, water supply, year; To..."
4,Electrical Engineering & Computer Science,6.60,"state, management, development",0,0,,


## 11. Map topic keyword groups onto the backbone and project documents

This produces simple counts showing which topic groups appear in each document and how those groups project onto the filtered science backbone.


In [42]:
keyword_groups = {
    sbp.topic_display_name(topic_data): list(topic_data.get("keywords", []) or [])
    for _, topic_data in topics_info.items()
}

keyword_group_mappings = sbp.map_keyword_groups_to_backbone(
    keyword_groups,
    science_backbone,
    top_n=3,
    as_dataframe=True,
)
keyword_group_mappings.to_csv(OUTPUT_DIR / "keyword_group_backbone_mappings.csv", index=False)

documents_by_name = {
    name: text
    for name, text in zip(doc_names, processed_docs)
}

backbone_projection = sbp.project_documents_onto_backbone(
    documents=documents_by_name,
    keyword_groups=keyword_groups,
    group_mappings=keyword_group_mappings,
)

for table_name, table_df in backbone_projection.items():
    path = OUTPUT_DIR / f"{table_name}.csv"
    table_df.to_csv(path, index=False)
    print(f"Wrote {table_name}: {path}")

display(keyword_group_mappings)
display(backbone_projection["domain_totals"])


Wrote doc_group_counts: /Users/wmobley/Documents/GitHub/DSO/DSO-Institute-2026/Day-2/Morning/outputs/doc_group_counts.csv
Wrote group_totals: /Users/wmobley/Documents/GitHub/DSO/DSO-Institute-2026/Day-2/Morning/outputs/group_totals.csv
Wrote domain_totals: /Users/wmobley/Documents/GitHub/DSO/DSO-Institute-2026/Day-2/Morning/outputs/domain_totals.csv
Wrote subdiscipline_totals: /Users/wmobley/Documents/GitHub/DSO/DSO-Institute-2026/Day-2/Morning/outputs/subdiscipline_totals.csv


,group,primary_domain,backbone_mapping,matched_terms,n_matches
0,"Topic 1: surface water, water supply, year","Chemical, Mechanical, & Civil Engineering","[(Chemical, Mechanical, & Civil Engineering, E...","{'Chemical, Mechanical, & Civil Engineering::E...",0
1,"Topic 2: carrizo, subsidence, water","Chemical, Mechanical, & Civil Engineering","[(Chemical, Mechanical, & Civil Engineering, E...","{'Chemical, Mechanical, & Civil Engineering::E...",0
2,"Topic 3: water, state, groundwater","Chemical, Mechanical, & Civil Engineering","[(Chemical, Mechanical, & Civil Engineering, E...","{'Chemical, Mechanical, & Civil Engineering::E...",0
3,"Topic 4: regulatory, district, plan",Biology,"[(Biology, Biological Conservation, 2), (Biolo...","{'Biology::Biological Conservation': ['plan', ...",0
4,"Topic 5: gcd, groundwater, carrizo","Chemical, Mechanical, & Civil Engineering","[(Chemical, Mechanical, & Civil Engineering, E...","{'Chemical, Mechanical, & Civil Engineering::E...",0


,domain,weight
0,"Chemical, Mechanical, & Civil Engineering",27359.0
1,Biology,6730.0


## 12. Extract decision components

This uses the package defaults and writes tables for downstream review. If your local package signature differs, the `try` block will print the error without stopping the rest of the notebook.


In [ ]:
try:
    components = sbp.extract_decision_components(transcripts, nlp)
    components_df = sbp.component_table(components)
    components_path = OUTPUT_DIR / "decision_components.csv"
    components_df.to_csv(components_path, index=False)
    print(f"Wrote decision components: {components_path}")
    display(components_df.head(25))

    try:
        component_fig = sbp.plot_component_distribution(sbp.component_counts(components))
        component_fig.show()
    except Exception as plot_exc:
        print(f"Skipping component plot: {type(plot_exc).__name__}: {plot_exc}")
except Exception as exc:
    components = {}
    components_df = pd.DataFrame()
    print(f"Skipping decision-component extraction: {type(exc).__name__}: {exc}")


## 13. Create SVO / variable mappings

The scientific-variable vocabulary comes from MINT's `variablepresentations` endpoint when `MINT_API_BASE_URL` is configured — each MINT variable contributes its standard name, units, domain, and keyword aliases. If MINT is unavailable, the cell falls back to a small built-in vocabulary so the rest of the notebook still runs.


In [ ]:
try:
    if MINT_API_BASE_URL:
        try:
            svo_vocabulary = sbp.fetch_mint_svo_vocabulary(
                base_url=MINT_API_BASE_URL,
                username=MINT_USERNAME or "mint@isi.edu",
                per_page=MINT_SVO_PER_PAGE,
                max_pages=MINT_SVO_MAX_PAGES,
            )
            print(f"Loaded {len(svo_vocabulary)} variables from MINT ({MINT_API_BASE_URL})")
        except Exception as mint_exc:
            print(
                f"MINT vocabulary fetch failed ({type(mint_exc).__name__}: {mint_exc}); "
                f"falling back to default SVO vocabulary."
            )
            svo_vocabulary = sbp.default_svo_vocabulary()
    else:
        svo_vocabulary = sbp.default_svo_vocabulary()
        print(
            f"MINT_API_BASE_URL not set; using built-in default SVO vocabulary "
            f"({len(svo_vocabulary)} variables)."
        )

    # MINT and default vocabularies both include single-word keywords (e.g. "aquifer",
    # "groundwater"). Allow them through to match what MINT actually exposes.
    svo_mappings = sbp.create_svo_mappings(
        documents=transcripts,
        svo_vocabulary=svo_vocabulary,
        min_keyword_words=1,
    )
    svo_mappings = sbp.deduplicate_svo_mappings(svo_mappings)
    svo_df = sbp.svo_table(svo_mappings)
    svo_path = OUTPUT_DIR / "svo_mappings.csv"
    svo_df.to_csv(svo_path, index=False)
    print(f"Wrote SVO mappings: {svo_path} ({len(svo_mappings)} rows)")
    display(svo_df.head(25))

    if not svo_mappings:
        print("No SVO mappings found in this corpus; skipping sunburst plot.")
    else:
        try:
            svo_sunburst_df, svo_fig, svo_domain_counts = sbp.plot_svo_sunburst(svo_mappings)
            svo_fig.show()
        except Exception as plot_exc:
            print(f"Skipping SVO plot: {type(plot_exc).__name__}: {plot_exc}")
except Exception as exc:
    svo_mappings = []
    svo_df = pd.DataFrame()
    print(f"Skipping SVO mapping: {type(exc).__name__}: {exc}")


## 14. Build a summary report


In [45]:
try:
    report = sbp.build_summary_report(
        case_study_name=CASE_STUDY_NAME,
        transcripts=transcripts,
        n_topics=n_topics,
        topic_mappings=topic_mappings,
        decision_components=components,
        unique_mappings=svo_mappings,
        svo_df=svo_df,
    )
    report_path = sbp.write_report(OUTPUT_DIR, CASE_STUDY_NAME, report)
    print(f"Wrote report: {report_path}")
except Exception as exc:
    print(f"Skipping report generation: {type(exc).__name__}: {exc}")


Wrote report: /Users/wmobley/Documents/GitHub/DSO/DSO-Institute-2026/Day-2/Morning/outputs/Groundwater Management Area 12_report.md


## 15. Output checklist

Expected files in `outputs/` include:

- `science_backbone.json`
- `topic_mappings.csv`
- `semantic_bridge_network.html`
- `keyword_group_backbone_mappings.csv`
- `doc_group_counts.csv`
- `domain_totals.csv`
- `subdiscipline_totals.csv`
- optionally `decision_components.csv`, `svo_mappings.csv`, and `semantic_bridge_report.md`


In [46]:
print("Output files:")
for path in sorted(OUTPUT_DIR.glob("*")):
    if path.is_file():
        print(f"  - {path.name}")


Output files:
  - Groundwater Management Area 12_report.md
  - UCSDmap_with_disciplines.net.txt
  - decision_components.csv
  - doc_group_counts.csv
  - domain_totals.csv
  - group_totals.csv
  - keyword_group_backbone_mappings.csv
  - mint_topic_recommendations.csv
  - science_backbone.json
  - semantic_bridge_pyvis_graph_2d.html
  - subdiscipline_totals.csv
  - svo_mappings.csv
  - topic_mappings.csv
